In [1]:
# Import libraries
import pandas as pd

In [14]:
# Load the collision data
# Use the sample dataset present in the script and save it in the same
collisions = pd.read_csv("sample_collisions.csv")
print("Collisions data loaded:", collisions.shape)

# Load the parties (drivers/passengers) data
parties = pd.read_csv("sample_parties.csv")
print("Parties data loaded:", parties.shape)

Collisions data loaded: (935791, 65)
Parties data loaded: (1866917, 24)


In [3]:
# Preview data
display(collisions.head())
display(parties.head())

,case_id,jurisdiction,officer_id,reporting_district,chp_shift,population,county_city_location,county_location,special_condition,beat_type,...,pedestrian_injured_count,bicyclist_killed_count,bicyclist_injured_count,motorcyclist_killed_count,motorcyclist_injured_count,latitude,longitude,collision_date,collision_time,process_date
0,8000993.0,1942.0,39335,1676,not chp,>250000,1942,los angeles,0.0,not chp,...,0,0,0,0,0.0,NaN,NaN,2016-03-20,16:30:00,2016-03-30
1,4232117.0,3008.0,1284,Z2,not chp,100000 to 250000,3008,orange,0.0,not chp,...,0,0,0,0,0.0,NaN,NaN,2009-04-13,18:24:00,2009-11-30
2,91389627.0,9285.0,019223,NaN,1400 thru 2159,unincorporated,5800,yuba,0.0,chp county roadline,...,0,0,0,0,0.0,39.12512,-121.57211,2021-01-09,18:15:00,2021-01-18
3,3589817.0,3610.0,50586,3610,not chp,100000 to 250000,3610,san bernardino,0.0,not chp,...,0,0,0,0,0.0,NaN,NaN,2007-06-16,21:50:00,2008-03-11
4,9111695.0,1900.0,649318,0261,not chp,25000 to 50000,1917,los angeles,0.0,not chp,...,0,0,0,0,0.0,NaN,NaN,2020-04-14,17:56:00,2020-08-03


,id,case_id,party_number,party_type,at_fault,party_sex,party_age,party_sobriety,direction_of_travel,party_safety_equipment_1,...,other_associate_factor_1,party_number_killed,party_number_injured,movement_preceding_collision,vehicle_year,vehicle_make,statewide_vehicle_type,chp_vehicle_type_towing,chp_vehicle_type_towed,party_race
0,1292305,4.843348e+06,1,driver,1,male,28.0,had not been drinking,south,lap/shoulder harness used,...,inattention,0,0,making right turn,1974.0,chevrolet,passenger car,NaN,NaN,NaN
1,3252679,5.965300e+06,2,driver,0,male,53.0,had not been drinking,south,air bag not deployed,...,entering/leaving ramp,0,0,proceeding straight,2000.0,toyota,passenger car,"passenger car, station",00,hispanic
2,5367929,7.110929e+06,2,driver,0,female,22.0,had not been drinking,west,air bag not deployed,...,none apparent,0,0,proceeding straight,2008.0,mazda,passenger car,"passenger car, station",NaN,hispanic
3,1482458,4.945119e+06,2,pedestrian,0,female,52.0,had not been drinking,south,NaN,...,none apparent,0,1,proceeding straight,NaN,NaN,pedestrian,pedestrian,00,hispanic
4,16036906,9.250011e+18,2,driver,0,female,34.0,had not been drinking,south,lap/shoulder harness used,...,none apparent,0,0,proceeding straight,1976.0,ford,passenger car,"passenger car, station",00,NaN


In [ ]:
# Transform

In [5]:
# 1. Select key columns from both datasets
collisions_filtered = collisions[
    [
        "case_id",
        "county_city_location",
        "county_location",
        "special_condition",
        "pedestrian_injured_count",
        "bicyclist_killed_count",
        "motorcyclist_injured_count",
        "latitude",
        "longitude",
        "collision_date",
        "collision_time",
    ]
]

parties_filtered = parties[
    [
        "case_id",
        "party_number",
        "party_type",
        "at_fault",
        "party_sex",
        "party_age",
        "party_sobriety",
        "direction_of_travel",
        "party_safety_equipment_1",
        "party_number_killed",
        "party_number_injured",
        "vehicle_year",
        "vehicle_make",
        "statewide_vehicle_type",
        "party_race",
    ]
]

In [6]:
# 2. Convert data types
collisions_filtered["collision_date"] = pd.to_datetime(
    collisions_filtered["collision_date"], errors="coerce"
)
parties_filtered["party_age"] = pd.to_numeric(
    parties_filtered["party_age"], errors="coerce"
)

/tmp/ipykernel_785199/329207648.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  collisions_filtered['collision_date'] = pd.to_datetime(collisions_filtered['collision_date'], errors='coerce')
/tmp/ipykernel_785199/329207648.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  parties_filtered['party_age'] = pd.to_numeric(parties_filtered['party_age'], errors='coerce')


In [7]:
# 3. Filter to only drivers (party_type == 1 typically indicates driver)
drivers = parties_filtered[parties_filtered["party_type"] == 1]

In [8]:
# 4. Merge datasets on `case_id`
merged_data = pd.merge(collisions_filtered, drivers, on="case_id", how="inner")
print("Merged dataset shape:", merged_data.shape)

Merged dataset shape: (0, 25)


In [9]:
# 5. Add derived columns (example: age group)
merged_data["age_group"] = pd.cut(
    merged_data["party_age"],
    bins=[0, 18, 30, 45, 60, 100],
    labels=["<18", "18-30", "31-45", "46-60", "60+"],
)

In [ ]:
# LOAD

In [10]:
# Save the cleaned and joined dataset to a new CSV
merged_data.to_csv("merged_traffic_data.csv", index=False)
print("Final data saved to 'merged_traffic_data.csv'")

Final data saved to 'merged_traffic_data.csv'


In [11]:
# Real-World Use Case: Driver Age vs Sobriety

In [12]:
# Count of collisions by driver age group and sobriety status
age_sobriety_analysis = (
    merged_data.groupby(["age_group", "party_sobriety"])
    .size()
    .unstack()
    .fillna(0)
    .astype(int)
)

/tmp/ipykernel_785199/3321812912.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  merged_data.groupby(['age_group', 'party_sobriety'])


In [13]:
# Display result
print("Collision count by age group and sobriety status:")
display(age_sobriety_analysis)

Collision count by age group and sobriety status:


party_sobriety
age_group
